In [1]:
import csv
import json
import os
import shutil
from collections import Counter

SEED = 9
TARGET_YEAR = str(2015 + SEED)

input_file= "Global_Cybersecurity_Threats_2015-2024.csv"
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

filtered_csv_path = os.path.join(output_dir, "filtered_incidents.csv")
json_summary_path = os.path.join(output_dir, "cybersecurity_summary.json")
backup_json_path = os.path.join(output_dir, "cybersecurity_summary_backup.json")
text_report_path = os.path.join(output_dir, "cybersecurity_text_report.txt")

total_records = 0
target_year_count = 0
attack_type_counts = Counter()
total_financial_loss = 0.0
target_incidents = []

print(f"Processing cybersecurity incidents for the year {TARGET_YEAR}...")

Processing cybersecurity incidents for the year 2024...


In [2]:
with open(input_file, mode="r", encoding="utf-8", errors="ignore") as f:
    reader = csv.reader(f)
    header = next(reader, None)

    for row in reader:
        total_records += 1
        try:
            date_field = row[1].strip()
            attack_type = row[2].strip()
            try:
                financial_loss = float(row[3].strip())
            except (ValueError, TypeError):
                continue
            if TARGET_YEAR in date_field:
                target_year_count += 1
                attack_type_counts[attack_type] += 1
                total_financial_loss += financial_loss

                target_incidents.append({
                    "row_data": row,
                    "loss": financial_loss,
                    "attack_type": attack_type
                })

        except (IndexError, ValueError):
            continue

    if target_year_count > 0:
        avg_financial_loss = total_financial_loss / target_year_count
    else:
        avg_financial_loss = 0.0

    sorted_incidents = sorted(
        target_incidents, key=lambda x: x["loss"], reverse=True
    )
    top_5_costliest = sorted_incidents[:5]

    with open(filtered_csv_path, mode="w", newline="", encoding="utf-8") as f_out:
        writer = csv.writer(f_out)
        if header:
            writer.writerow(header)
        for inc in target_incidents:
            writer.writerow(inc["row_data"])

    summary_data = {
        "target_year": TARGET_YEAR,
        "total_records_in_dataset": total_records,
        "incidents_in_target_year": target_year_count,
        "count_per_attack_type": dict(attack_type_counts),
        "total_financial_loss_million_usd": round(total_financial_loss, 2),
        "average_financial_loss_million_usd": round(avg_financial_loss, 2),
        "five_costliest_incidents": [
            {"loss": inc["loss"], "attack_type": inc["attack_type"]}
            for inc in top_5_costliest
        ],
    }

    with open(json_summary_path, mode="w", encoding="utf-8") as j_out:
        json.dump(summary_data, j_out, indent=4)

    shutil.copy(json_summary_path, backup_json_path)
    backup_verified = os.path.exists(backup_json_path)

    with open(text_report_path, mode="w", encoding="utf-8") as r_out:
        r_out.write(f"CYBERSECURITY INCIDENT REPORT - YEAR {TARGET_YEAR}\n")
        r_out.write("=" * 50 + "\n")
        r_out.write(f"Total Records Processed: {total_records}\n")
        r_out.write(f"Incidents in Target Year: {target_year_count}\n")
        r_out.write(
            f"Total Financial Loss: ${total_financial_loss:,.2f} Million USD\n"
        )
        r_out.write(
            f"Average Financial Loss: ${avg_financial_loss:,.2f} Million USD\n"
        )
        r_out.write("\nAttack Type Breakdown:\n")
        for attack, count in attack_type_counts.items():
            r_out.write(f" - {attack}: {count}\n")

In [10]:
print(f"\n--- Processing Complete ---")
print(f"Target Year: {TARGET_YEAR}")
print(f"Total records in dataset: {total_records}")
print(f"Incidents in target year: {target_year_count}")
print(f"Count per attack type (top 3): {dict(attack_type_counts.most_common(3))}")
print(f"Total financial loss (USD millions): {total_financial_loss:,.2f}")
print(f"Five costliest incidents: {len(top_5_costliest)} incidents")
print(f"Filtered CSV file: {filtered_csv_path}")
print(f"JSON summary file: {json_summary_path}")
print(f"Backup verification: {backup_verified}")
print(f"Report file: {text_report_path}")


--- Processing Complete ---
Target Year: 2024
Total records in dataset: 3000
Incidents in target year: 0
Count per attack type (top 3): {}
Total financial loss (USD millions): 0.00
Five costliest incidents: 0 incidents
Filtered CSV file: output\filtered_incidents.csv
JSON summary file: output\cybersecurity_summary.json
Backup verification: True
Report file: output\cybersecurity_text_report.txt
